# RE-THREAD: Care Label Detection Training (YOLOv8 Single Class 'tag')

**Team Circular Transformation · PCCOE Pune**  
**Objective:** Fine-tune Ultralytics YOLOv8 (`yolov8n.pt`) to localize the care label in a garment photograph and return its crop coordinates for OpenCV preprocessing and OCR.

---

### Why localize the tag?
Care labels are small, low-contrast, and often rotated relative to the entire garment. Passing a full high-resolution garment photo directly to OCR produces noise and poor accuracy. Cropping the care label first increases OCR precision from ~40% to >90%.

## Step 1: Environment Setup & GPU Verification

In [ ]:
!nvidia-smi

import torch
print(f"CUDA active: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive & Install Ultralytics

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/rethread_weights'
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install --upgrade ultralytics
import ultralytics
ultralytics.checks()

## Step 3: Dataset Configuration (Single Class: 'tag')

Prepare your `data.yaml` pointing to train and val images:
```yaml
path: /content/tag_dataset
train: images/train
val: images/val
names:
  0: tag
```

In [ ]:
# If using Roboflow care label dataset:
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# project = rf.workspace("care-labels").project("garment-tag-detect")
# dataset = project.version(1).download("yolov8")
# DATA_YAML = f"{dataset.location}/data.yaml"

DATA_YAML = "/content/tag_dataset/data.yaml"
print(f"Target data.yaml: {DATA_YAML}")

## Step 4: Train YOLOv8 on Care Label Localization
Using pretrained `yolov8n.pt` (nano detector, fast on CPU/GPU).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=40,
    imgsz=640,
    batch=16,
    project="rethread_training",
    name="tag_yolov8_detect",
    save=True,
)
print("Training complete!")

## Step 5: Save Weights to Google Drive as `tag.pt`

In [ ]:
import shutil

best_weights = "rethread_training/tag_yolov8_detect/weights/best.pt"
target_drive_file = f"{OUTPUT_DIR}/tag.pt"

shutil.copy(best_weights, target_drive_file)
print(f"SUCCESS: Saved trained tag detector weights to:\n{target_drive_file}")
print("\n--- NEXT STEP ---")
print("Download tag.pt and place into: re-thread/weights/tag.pt")